In [1]:
import pandas as pd
import numpy as np
import re
from difflib import SequenceMatcher

In [2]:
# 1. Đọc dữ liệu
df_transformed = pd.read_csv('data/transformed_dataset.csv')
df_mapping = pd.read_csv('data/symptoms_mapping.csv')

# Làm sạch tên cột (loại bỏ khoảng trắng thừa)
symptom_cols = [col.strip() for col in df_transformed.columns if col not in ['Unnamed: 0', 'Disease', 'Disease_code']]
df_transformed.columns = [c.strip() for c in df_transformed.columns]

# 2. Tính toán trọng số (Weights) cho mỗi triệu chứng từ dataset
# Ta lấy giá trị phổ biến nhất (mode) của các ô có giá trị > 0 cho mỗi cột triệu chứng
weights_dict = {}
for col in symptom_cols:
    non_zero = df_transformed[col][df_transformed[col] > 0]
    if not non_zero.empty:
        weights_dict[col] = int(non_zero.mode()[0])
    else:
        weights_dict[col] = 1 # Mặc định nếu không có dữ liệu

# 3. Hàm so khớp mờ (Mô phỏng thefuzz)
def fuzzy_match_score(s1, s2):
    return SequenceMatcher(None, s1, s2).ratio()

# 4. Thuật toán tách từ và ánh xạ triệu chứng
def process_clinical_text(text, mapping_df, threshold=0.85):
    # Tiền xử lý văn bản
    text_clean = text.lower()
    text_clean = re.sub(r'[,.\?!\(\)]', ' ', text_clean)
    
    detected_en_symptoms = []
    
    # Duyệt qua bảng ánh xạ
    for _, row in mapping_df.iterrows():
        vn_term = str(row['vi']).lower().strip()
        en_term = row['en'].strip()
        
        # Bước A: Kiểm tra khớp chính xác theo cụm từ (Tránh khớp nhầm "đầu" trong "bắt đầu")
        if re.search(r'\b' + re.escape(vn_term) + r'\b', text_clean):
            detected_en_symptoms.append(en_term)
            continue
            
        # Bước B: So khớp mờ (Fuzzy) cho các biến thể từ ngữ
        words = text_clean.split()
        for i in range(len(words)):
            for length in [1, 2, 3]: # Xét từ đơn, từ ghép 2-3 chữ
                if i + length <= len(words):
                    sub_phrase = " ".join(words[i:i+length])
                    if fuzzy_match_score(sub_phrase, vn_term) > threshold:
                        detected_en_symptoms.append(en_term)
                        break
                        
    return list(set(detected_en_symptoms))

# 5. Chuyển đổi sang Ma trận (Vector hóa)
def convert_to_matrix(detected_list, all_features, weights):
    # Khởi tạo vector 0 với độ dài bằng số lượng triệu chứng trong dataset
    vector = np.zeros(len(all_features))
    
    for sym in detected_list:
        if sym in all_features:
            idx = all_features.index(sym)
            # Gán trọng số tương ứng từ weights_dict
            vector[idx] = weights.get(sym, 1)
            
    return vector

In [3]:
# --- THỰC THI ---
input_text = "Khoảng 3 ngày nay tôi cảm thấy cơ thể rất mệt mỏi. Da tôi bắt đầu xuất hiện những nốt phát ban đỏ và cảm thấy rất ngứa. Ngoài ra, tôi còn bị nổi mụn mủ nhỏ ở tay. Tôi cũng bị đau khớp và cảm thấy hơi lo lắng."

# B1: Trích xuất triệu chứng
detected = process_clinical_text(input_text, df_mapping)

# B2: Tạo ma trận số
input_vector = convert_to_matrix(detected, symptom_cols, weights_dict)

# B3: Xuất kết quả dưới dạng DataFrame để quan sát ma trận nhỏ (chỉ các cột có giá trị)
result_matrix = pd.DataFrame([input_vector], columns=symptom_cols)

# Lưu ma trận kết quả ra file csv
result_matrix.to_csv('data/transformed_input.csv', index=False)